# 03. Подготовка данных для моделирования

Здесь фиксируем окончательный набор атрибутов, проверяем их математические свойства и убеждаемся, что каждая пара муниципалитет-месяц уникальна. Этот файл используется непосредственно при построении сети.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Загрузка панели признаков

Берем результат `02_feature_engineering`.

In [2]:
features = pd.read_parquet(
    "../data/processed/features_basic.parquet"
)

features.head()

,territory_id,date,health,marketplaces,catering,food,transport,total_spending,food_share,health_share,marketplaces_share,catering_share,transport_share,market_access
0,1,2023-01,1271,2505,1142,7692,1718,14328,0.536851,0.088707,0.174832,0.079704,0.119905,309.6
1,1,2023-02,1276,2748,1095,8259,1704,15082,0.547606,0.084604,0.182204,0.072603,0.112982,309.6
2,1,2023-03,1410,3052,1256,8873,1847,16438,0.539786,0.085777,0.185667,0.076408,0.112362,309.6
3,1,2023-04,1277,2850,1171,8648,1784,15730,0.549777,0.081182,0.181182,0.074444,0.113414,309.6
4,1,2023-05,1295,2994,1315,8738,1817,16159,0.540751,0.080141,0.185284,0.081379,0.112445,309.6


In [3]:
features.columns.tolist()

['territory_id',
 'date',
 'health',
 'marketplaces',
 'catering',
 'food',
 'transport',
 'total_spending',
 'food_share',
 'health_share',
 'marketplaces_share',
 'catering_share',
 'transport_share',
 'market_access']

## 2. Финальный набор признаков модели

Для экономической близости используем пять долей потребления. Показатель `market_access` оставляем вне основного вектора.

In [4]:
model_features = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

## 3. Описательная статистика

Проверяем диапазоны и распределения признаков перед построением расстояний.

In [5]:
features[model_features].describe()

,food_share,health_share,marketplaces_share,catering_share,transport_share
count,50521.000000,50521.000000,50521.000000,50521.000000,50521.000000
mean,0.610033,0.068492,0.190644,0.048509,0.082322
std,0.076177,0.015779,0.057837,0.036424,0.023919
min,0.291669,0.007437,0.010473,0.000773,0.006377
25%,0.561653,0.057455,0.151774,0.020981,0.064535
50%,0.617575,0.067066,0.189534,0.035695,0.078958
75%,0.661606,0.078297,0.227584,0.067514,0.099578
max,0.906136,0.168341,0.531325,0.215699,0.225053


## 4. Проверка пропусков

Все пять признаков, которые войдут в расстояние между муниципалитетами, должны быть заполнены.

In [6]:
features[model_features].isna().sum()

food_share            0
health_share          0
marketplaces_share    0
catering_share        0
transport_share       0
dtype: int64

## 5. Проверка суммы долей

Так как признаки являются долями выбранных категорий, их сумма должна быть равна 1 с учетом численной погрешности.

In [7]:
features["share_sum"] = features[model_features].sum(axis=1)

features["share_sum"].describe()

count    5.052100e+04
mean     1.000000e+00
std      6.521944e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: share_sum, dtype: float64

In [8]:
(features["share_sum"] - 1).abs().max()

np.float64(2.220446049250313e-16)

## 6. Нормализация даты

Приводим дату к типу datetime, чтобы корректно фильтровать месяцы и строить динамическую модель.

In [9]:
features["date"] = pd.to_datetime(features["date"])

features["date"].head()

0   2023-01-01
1   2023-02-01
2   2023-03-01
3   2023-04-01
4   2023-05-01
Name: date, dtype: datetime64[us]

## 7. Проверка уникальности municipality-month

Для каждого муниципалитета и месяца должна существовать одна строка с экономическим профилем.

In [10]:
duplicates = features.duplicated(
    subset=["territory_id", "date"]
).sum()

print("Дубликатов муниципалитет-месяц:", duplicates)

Дубликатов муниципалитет-месяц: 0


## 8. Проверка числа муниципалитетов по месяцам

Смотрим, насколько одинаково сформирована панель по времени.

In [11]:
monthly_counts = (
    features
    .groupby("date")["territory_id"]
    .nunique()
)

monthly_counts

date
2023-01-01    2143
2023-02-01    2141
2023-03-01    2149
2023-04-01    2129
2023-05-01    2121
2023-06-01    2119
2023-07-01    2126
2023-08-01    2118
2023-09-01    2109
2023-10-01    2103
2023-11-01    2093
2023-12-01    2103
2024-01-01    2072
2024-02-01    2078
2024-03-01    2083
2024-04-01    2087
2024-05-01    2085
2024-06-01    2084
2024-07-01    2089
2024-08-01    2094
2024-09-01    2100
2024-10-01    2092
2024-11-01    2100
2024-12-01    2103
Name: territory_id, dtype: int64

## 9. Финальная проверка периода

Фиксируем список месяцев, который станет основой динамического моделирования.

In [12]:
months = sorted(features["date"].unique())

len(months), months[:3], months[-3:]

(24,
 [Timestamp('2023-01-01 00:00:00'),
  Timestamp('2023-02-01 00:00:00'),
  Timestamp('2023-03-01 00:00:00')],
 [Timestamp('2024-10-01 00:00:00'),
  Timestamp('2024-11-01 00:00:00'),
  Timestamp('2024-12-01 00:00:00')])